# Proyecto 2 - Extracción de Características

## 02_Feature_Engineering.ipynb

Este notebook crea nuevas variables a partir de los datos climáticos y prepara el dataset para modelos de clasificación y series de tiempo.

### Objetivos
- Extraer características temporales de la columna `date`
- Calcular indicadores como razón humedad/presión y promedios móviles
- Generar etiquetas de clasificación de temperatura
- Guardar un dataset preparado para modelado

In [ ]:
import pandas as pd
import numpy as np

train_path = '../data/DailyDelhiClimateTrain.csv'
test_path = '../data/DailyDelhiClimateTest.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

## Creación de características temporales

In [ ]:
for df in [train_df, test_df]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

train_df[['date', 'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend']].head()

## Nuevas variables climáticas

In [ ]:
for df in [train_df, test_df]:
    df['humidity_pressure_ratio'] = df['humidity'] / df['meanpressure']
    df['temp_diff'] = df['meantemp'].diff().fillna(0)
    df['pressure_diff'] = df['meanpressure'].diff().fillna(0)
    df['humidity_diff'] = df['humidity'].diff().fillna(0)

train_df[['humidity_pressure_ratio', 'temp_diff', 'pressure_diff', 'humidity_diff']].head()

## Promedios móviles y ventanas temporales

In [ ]:
window_cols = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
for df in [train_df, test_df]:
    for col in window_cols:
        df[f'{col}_rolling_3'] = df[col].rolling(window=3, min_periods=1).mean()
        df[f'{col}_rolling_7'] = df[col].rolling(window=7, min_periods=1).mean()

train_df[[col for col in train_df.columns if 'rolling' in col]].head()

## Etiquetas de clasificación de temperatura

Convertimos la variable `meantemp` en una categoría sencilla para entrenar modelos de clasificación.

In [ ]:
labels = [
    (train_df['meantemp'] < 15),
    (train_df['meantemp'] >= 15) & (train_df['meantemp'] < 25),
    (train_df['meantemp'] >= 25)
]
choices = ['cold', 'mild', 'hot']
train_df['temp_category'] = np.select(labels, choices)
train_df['temp_category'].value_counts()

## Preparación final de columnas para modelado

In [ ]:
feature_columns = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend',
    'humidity', 'wind_speed', 'meanpressure', 'humidity_pressure_ratio',
    'meantemp_rolling_3', 'meantemp_rolling_7',
    'humidity_rolling_3', 'humidity_rolling_7',
    'wind_speed_rolling_3', 'wind_speed_rolling_7',
    'meanpressure_rolling_3', 'meanpressure_rolling_7'
]

train_df[feature_columns].head()

## Guardar dataset preparado

Guardamos los datos listos para modelado en el directorio `notebooks`.

In [ ]:
train_df.to_csv('../data/train_feature_engineered.csv', index=False)
test_df.to_csv('../data/test_feature_engineered.csv', index=False)
print('Archivos guardados: train_feature_engineered.csv, test_feature_engineered.csv')